# 📊 Fase 2: Auditoría de Calidad y Análisis Exploratorio de Datos (EDA)
## Proyecto de Ciencia de Datos - Inside Airbnb (New York City)

---

### 🎯 Objetivo del Notebook
Realizar una auditoría exhaustiva de la calidad de los datos crudos de **Inside Airbnb NYC**, diagnosticando problemas reales de "suciedad" (inconsistencias de tipos, valores monetarios en texto, listas anidadas, outliers y patrones de nulos), y explorar las relaciones bivariadas con el precio por noche para sentar las bases del modelado predictivo (Fase 3 y 4).

---

In [ ]:
# 1. Importación de Librerías
import sys
from pathlib import Path

# Añadir src al path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

# Configuración visual
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 11

print("✅ Entorno y librerías cargadas correctamente.")


---
## 2. Carga del Dataset Crudo
Cargamos el dataset comprimido `listings.csv.gz` de New York City.

In [ ]:
from src.data_loader import load_raw_listings

# Carga de datos de NYC
df_raw = load_raw_listings(city="nyc")
print(f"Dimensiones crudas: {df_raw.shape[0]:,} filas x {df_raw.shape[1]} columnas")
df_raw.head(3)


---
## 3. Diagnóstico de Calidad de Datos (Data Quality Audit)

### 3.1 Auditoría de Valores Nulos
Analizamos qué columnas tienen nulos y clasificamos su significado de negocio (ej. propiedades sin reseñas vs. campos no capturados).

In [ ]:
# Auditoría de Nulos
null_series = df_raw.isnull().mean() * 100
null_summary = pd.DataFrame({
    'Columna': null_series.index,
    'Porcentaje_Nulos': null_series.values,
    'Total_Nulos': df_raw.isnull().sum().values
}).sort_values(by='Porcentaje_Nulos', ascending=False)

# Columnas con 100% de nulos (candidatas a eliminación inmediata)
cols_100_null = null_summary[null_summary['Porcentaje_Nulos'] == 100]['Columna'].tolist()
print(f"⚠️ Columnas con 100% de nulos ({len(cols_100_null)} columnas):")
print(cols_100_null)

# Visualización del Top 25 columnas con más nulos (excluyendo 100% vacías)
top_nulls = null_summary[(null_summary['Porcentaje_Nulos'] > 0) & (null_summary['Porcentaje_Nulos'] < 100)].head(25)

plt.figure(figsize=(12, 7))
sns.barplot(data=top_nulls, x='Porcentaje_Nulos', y='Columna', hue='Columna', palette='viridis', legend=False)
plt.title('Top 25 Columnas con Valores Faltantes (0% < Nulos < 100%)', fontsize=14, fontweight='bold')
plt.xlabel('% de Valores Nulos')
plt.ylabel('Variable')
plt.tight_layout()
plt.show()


---
### 3.2 Limpieza y Parseo de Tipos de Datos Sucios
En el dataset crudo:
- `price` viene como texto (`"$150.00"`).
- `bathrooms_text` contiene texto mixto (`"1.5 baths"`, `"Shared half-bath"`).
- `amenities` viene como un string con sintaxis de lista JSON.

In [ ]:
def parse_price(val):
    """Convierte strings monetarios como '$125.00' a float."""
    if pd.isna(val):
        return np.nan
    if isinstance(val, (int, float)):
        return float(val)
    val_clean = re.sub(r'[^\d.]', '', str(val))
    try:
        return float(val_clean)
    except ValueError:
        return np.nan

def parse_bathrooms(val):
    """Extrae el número flotante de strings como '1.5 baths' o 'Shared half-bath'."""
    if pd.isna(val):
        return np.nan
    val_str = str(val).lower()
    if 'half' in val_str:
        return 0.5
    match = re.search(r'(\d+(\.\d+)?)', val_str)
    return float(match.group(1)) if match else np.nan

# Aplicamos transformaciones iniciales sobre una copia de trabajo
df = df_raw.copy()
df['price_clean'] = df['price'].apply(parse_price)
df['bathrooms_clean'] = df['bathrooms_text'].apply(parse_bathrooms)

print("Muestra de precios parseados:")
print(df[['price', 'price_clean', 'bathrooms_text', 'bathrooms_clean']].dropna().head(5))


---
### 3.3 Análisis de Valores Atípicos (Outliers) en la Variable Target (`price`)
El precio por noche presenta una distribución asimétrica con colas pesadas. Evaluamos percentiles y precios anómalos.

In [ ]:
# Estadísticas descriptivas del precio limpio
price_desc = df['price_clean'].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99])
print("Resumen Estadístico de Precios por Noche ($USD):")
print(price_desc.apply(lambda x: f"${x:,.2f}"))

# Detección de precios cero o extremadamente altos
invalid_prices = df[(df['price_clean'] <= 0) | (df['price_clean'] > 2000)]
print(f"\n⚠️ Registros con precio <= 0 o > $2,000/noche: {len(invalid_prices)} ({len(invalid_prices)/len(df)*100:.2f}%)")

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Distribución cruda (filtrada a <= $1000 para visualización)
sns.histplot(df[df['price_clean'] <= 1000]['price_clean'], bins=50, kde=True, ax=axes[0], color='teal')
axes[0].set_title('Distribución de Precio por Noche (<= $1,000)', fontweight='bold')
axes[0].set_xlabel('Precio ($USD)')

# Transformación Logarítmica log(1 + price)
sns.histplot(np.log1p(df['price_clean'].dropna()), bins=50, kde=True, ax=axes[1], color='coral')
axes[1].set_title('Distribución de log(1 + Precio)', fontweight='bold')
axes[1].set_xlabel('log(1 + Precio)')

plt.tight_layout()
plt.show()


---
### 3.4 Desanidado y Frecuencia de Comodidades (`amenities`)
La columna `amenities` contiene una lista textual de servicios. Analizamos cuáles son las comodidades más frecuentes y su asociación con el precio.

In [ ]:
from collections import Counter

def extract_amenities_list(val):
    if pd.isna(val):
        return []
    try:
        items = json.loads(val)
        return [i.strip() for i in items if i.strip()]
    except Exception:
        # Fallback si no es JSON válido
        return [i.strip(' "'[]') for i in str(val).split(',') if i.strip(' "'[]')]

df['amenities_list'] = df['amenities'].apply(extract_amenities_list)

# Conteo de todas las comodidades
all_amenities = [item for sublist in df['amenities_list'] for item in sublist]
amenity_counts = Counter(all_amenities).most_common(20)

amenities_df = pd.DataFrame(amenity_counts, columns=['Comodidad', 'Frecuencia'])
amenities_df['Porcentaje_Listings'] = (amenities_df['Frecuencia'] / len(df)) * 100

plt.figure(figsize=(12, 6))
sns.barplot(data=amenities_df, x='Porcentaje_Listings', y='Comodidad', hue='Comodidad', palette='mako', legend=False)
plt.title('Top 20 Comodidades Más Frecuentes en NYC', fontsize=14, fontweight='bold')
plt.xlabel('% de Alojamientos que la ofrecen')
plt.ylabel('Comodidad')
plt.tight_layout()
plt.show()


---
## 4. Análisis Bivariado y Relaciones con el Negocio

### 4.1 Precio según Distrito (*Borough*) y Tipo de Habitación
¿Cómo varía el precio mediano por noche según el distrito de Nueva York y el tipo de alojamiento?

In [ ]:
# Filtramos outliers extremos para visualización clara ($10 - $1,500)
df_filtered = df[(df['price_clean'] >= 10) & (df['price_clean'] <= 1500)].copy()

plt.figure(figsize=(14, 6))
sns.boxplot(
    data=df_filtered,
    x='neighbourhood_group_cleansed',
    y='price_clean',
    hue='room_type',
    palette='Set2',
    showfliers=False
)
plt.title('Distribución de Precio por Distrito y Tipo de Habitación (NYC)', fontsize=14, fontweight='bold')
plt.xlabel('Distrito (Borough)')
plt.ylabel('Precio por Noche ($USD)')
plt.legend(title='Tipo de Habitación')
plt.tight_layout()
plt.show()

# Tabla de resumen de precios por Borough
borough_summary = df.groupby('neighbourhood_group_cleansed')['price_clean'].agg(
    Total_Listings='count',
    Precio_Mediano='median',
    Precio_Promedio='mean',
    Std_Dev='std'
).sort_values(by='Precio_Mediano', ascending=False)
print("Resumen de Precios por Distrito:")
print(borough_summary)


---
### 4.2 Matriz de Correlación Numérica con el Precio
Examinamos la correlación lineal y de rango entre las variables estructurales del inmueble (`accommodates`, `bedrooms`, `beds`, `bathrooms`, `number_of_reviews`, `review_scores_rating`) y el precio.

In [ ]:
numeric_cols = [
    'price_clean', 'accommodates', 'bedrooms_clean', 'beds', 'bathrooms_clean',
    'minimum_nights', 'maximum_nights', 'availability_30', 'availability_365',
    'number_of_reviews', 'review_scores_rating'
]

# Creamos una versión de bedrooms con fallback
df['bedrooms_clean'] = df['bedrooms']

corr_matrix = df[numeric_cols].corr(method='spearman')

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', cbar=True, vmin=-1, vmax=1)
plt.title('Matriz de Correlación de Spearman (Variables Clave vs Precio)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


---
## 5. Conclusiones de la Auditoría y Hallazgos para la Fase 3

### 📋 Hallazgos Principales:
1. **Target Asimétrico**: El precio por noche tiene una distribución fuertemente asimétrica hacia la derecha con outliers extremos. En la Fase 3 y 4 utilizaremos la transformación **$\log(1 + 	ext{price})$** o funciones de pérdida robustas (Huber / Fair Loss).
2. **Columnas a Descartar**: Hay variables con 100% de nulos en el scrape actual que deben eliminarse en el preprocesamiento (`neighbourhood`, `instant_bookable`, etc.).
3. **Poder Predictivo de Comodidades**: Las comodidades (*amenities*) tienen una cardinalidad alta pero las top 30 (como *Air Conditioning*, *Elevator*, *Dedicated workspace*, *Washer/Dryer*) explican diferenciales significativos en precio. Se generará un binarizador de comodidades clave en la Fase 3.
4. **Capacidad y Ubicación**: `accommodates`, `bathrooms_clean` y el distrito (`neighbourhood_group_cleansed` / `neighbourhood_cleansed`) son los predictores estructurales con mayor correlación con el precio.

---
### ⏭️ Próximo Paso:
Ejecutar el pipeline de **Fase 3 (Limpieza, Imputación e Ingeniería de Características)** en `notebooks/02_limpieza_feature_engineering.ipynb` y construir el `ColumnTransformer` en `src/cleaning.py`.
